# 실습 11: 조정 다이얼 찾기
- 상황: 모델에는 사람이 정해줘야 하는 값이 있는데, 지금까지 손대지 않고 썼다
- 목표: 그 값을 손으로 돌려보고, 자동 탐색으로 찾아본다

## Step 0. 앞 실습까지 재현하기

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = df.columns.drop("result")
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 다른 설정은 건드리지 않고 class_weight="balanced"만 추가한다
tree = DecisionTreeClassifier(class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)

예측 = tree.predict(X_test)

정확도 = (예측 == y_test).mean() * 100
맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()

print("정확도:", round(정확도, 2), "%")
print("잡은 불량:", 잡은불량)
print("놓친 불량:", 놓친불량)
print("헛경보:", 헛경보)

정확도: 89.17 %
잡은 불량: 4
놓친 불량: 17
헛경보: 17


학습용: 전체 1253건, 불량 83건<br>
시험용: 전체 314건, 불량 21건

---
## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 사람이 정해주는 값

| 말 | 뜻 |
|---|---|
| 하이퍼파라미터 | 학습으로 정해지지 않고 사람이 미리 정해줘야 하는 값. 설명서에 이 이름으로 나온다 |
| max_depth | 나무가 몇 번까지 갈라질 수 있는지. 스무고개를 몇 번까지 할 것인가 |
| min_samples_leaf | 갈라진 끝자리에 최소 몇 건은 있어야 하는지 |
| 자동 탐색 | 후보를 적어주면 조합마다 다 돌려보고 점수를 재는 것 |
| 기준(scoring) | 자동 탐색이 1등을 뽑을 때 쓰는 자. 정하지 않으면 정확도로 뽑는다 |

---

## Step 2. 깊이를 손으로 바꿔보기

In [12]:
# 나무 모델과 채점 도구를 불러온다
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

# 세 가지 깊이를 차례로 넣어본다. None 은 제한 없이 끝까지 간다는 뜻
for 깊이 in [3, 5, None]:
    # max_depth 자리만 바꾸고 나머지는 전부 같게 둔다
    나무 = DecisionTreeClassifier(random_state=42, class_weight="balanced", max_depth=깊이)
    나무.fit(X_train, y_train)
    예측 = 나무.predict(X_test)

    # ravel - 네 칸짜리 표를 한 줄로 펴서 이름을 하나씩 붙여 받는다
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()

    print(f"깊이 {깊이}: 정확도 {round((예측 == y_test).mean() * 100, 2)}%",
          f"| 잡은 불량 {잡은불량} 놓친 불량 {놓친불량} 헛경보 {헛경보}",
          f"| 재현율 {round(recall_score(y_test, 예측), 3)}",
          f"F1 {round(f1_score(y_test, 예측), 3)}")

깊이 3: 정확도 54.46% | 잡은 불량 12 놓친 불량 9 헛경보 134 | 재현율 0.571 F1 0.144
깊이 5: 정확도 68.15% | 잡은 불량 9 놓친 불량 12 헛경보 88 | 재현율 0.429 F1 0.153
깊이 None: 정확도 89.17% | 잡은 불량 4 놓친 불량 17 헛경보 17 | 재현율 0.19 F1 0.19


### 문법 노트 - 다이얼 돌리기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| max_depth=3 | 세 번까지만 갈라지게 한다 | 얕게 두면 잘게 외우지 못하고 뭉뚱그려 판단한다 |
| max_depth=None | 제한을 두지 않는다 | 기본값. 답이 나올 때까지 끝까지 갈라진다 |
| random_state=42 | 갈라지는 과정의 무작위 요소를 고정한다 | 다시 돌려도 같은 결과가 나오게 |

---
## Step 3. 깊이별 결과

| 깊이 | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|
| 3 | [54.46]% | [12] | [134] | [0.571] | [0.144] |
| 5 | [68.15]% | [9] | [88] | [0.429] | [0.153] |
| 제한 없음 | [89.17]% | [4] | [17] | [0.19] | [0.19] |

---
## Step 4. 자동 탐색으로 찾기

In [13]:
# GridSearchCV - 후보 조합을 전부 돌려보고 점수를 매겨주는 도구
from sklearn.model_selection import GridSearchCV

파라미터_후보 = {
    "max_depth": [2, 3, 4, 5, 10, None],
    "min_samples_leaf": [1, 5, 10, 20],
}

# scoring="recall" - 1등을 재현율로 뽑는다
# 학습용(X_train, y_train)만 넣는다 - 시험용은 여기서 전혀 쓰지 않는다
탐색 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=파라미터_후보,
    scoring="recall",
)
탐색.fit(X_train, y_train)

# 탐색이 고른 최적 모델로 시험용을 채점한다
최적모델 = 탐색.best_estimator_
예측_최적 = 최적모델.predict(X_test)

정확도 = (예측_최적 == y_test).mean() * 100
맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측_최적).ravel()

print("[1. 1등으로 뽑힌 설정값]")
print(" ", 탐색.best_params_)
print()

print("[2. 탐색 과정에서 나온 그 설정의 점수]")
print("  재현율(학습용 교차검증 평균):", round(탐색.best_score_, 2))
print()

결과표 = pd.DataFrame(탐색.cv_results_)
파라미터표 = pd.json_normalize(결과표["params"])
요약표 = pd.concat(
    [파라미터표, 결과표[["mean_test_score", "std_test_score", "rank_test_score"]]],
    axis=1,
)
요약표 = 요약표.rename(columns={
    "mean_test_score": "재현율(평균)",
    "std_test_score": "재현율(편차)",
    "rank_test_score": "순위",
})
요약표 = 요약표.sort_values("순위").head(5)

print("  탐색한 조합:", len(결과표), "개 / 상위 5개")
print(요약표.to_string(index=False))
print()

print("[3. 1등 설정으로 시험용", len(y_test), "건을 채점한 결과]")
print("  정확도:", round(정확도, 2), "%")
print("  잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("  재현율:", round(recall_score(y_test, 예측_최적), 3),
      "정밀도:", round(precision_score(y_test, 예측_최적, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 예측_최적), 3))

[1. 1등으로 뽑힌 설정값]
  {'max_depth': 3, 'min_samples_leaf': 20}

[2. 탐색 과정에서 나온 그 설정의 점수]
  재현율(학습용 교차검증 평균): 0.57

  탐색한 조합: 24 개 / 상위 5개
 max_depth  min_samples_leaf  재현율(평균)  재현율(편차)  순위
       3.0                20 0.565441 0.194713   1
       4.0                20 0.543382 0.167969   2
       3.0                 1 0.541912 0.175358   3
       3.0                10 0.541912 0.175358   3
       3.0                 5 0.541912 0.175358   3

[3. 1등 설정으로 시험용 314 건을 채점한 결과]
  정확도: 54.46 %
  잡은 불량: 12 / 놓친 불량: 9 / 헛경보: 134
  재현율: 0.571 정밀도: 0.082 F1: 0.144


---
## Step 5.

In [14]:
# 같은 후보(파라미터_후보), 같은 모델 - scoring만 "f1"로 바꾼다
F1_탐색 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    param_grid=파라미터_후보,
    scoring="f1",
)
F1_탐색.fit(X_train, y_train)

F1_최적모델 = F1_탐색.best_estimator_
F1_예측_최적 = F1_최적모델.predict(X_test)

F1_정확도 = (F1_예측_최적 == y_test).mean() * 100
F1_맞힌양품, F1_헛경보, F1_놓친불량, F1_잡은불량 = confusion_matrix(y_test, F1_예측_최적).ravel()

print("[1. 이번에 1등으로 뽑힌 설정값 (기준: F1)]")
print(" ", F1_탐색.best_params_)
print()

print("[2. 그 설정으로 시험용을 채점한 결과]")
print("  정확도:", round(F1_정확도, 2), "%")
print("  잡은 불량:", F1_잡은불량, "/ 헛경보:", F1_헛경보)
print("  재현율:", round(recall_score(y_test, F1_예측_최적), 3),
      "정밀도:", round(precision_score(y_test, F1_예측_최적, zero_division=0), 3),
      "F1:", round(f1_score(y_test, F1_예측_최적), 3))
print()

print("[3. 재현율 기준 vs F1 기준]")
비교표 = pd.DataFrame(
    [
        {
            "기준": "재현율",
            "설정값": 탐색.best_params_,
            "정확도(%)": round(정확도, 2),
            "잡은 불량": 잡은불량,
            "헛경보": 헛경보,
            "재현율": round(recall_score(y_test, 예측_최적), 3),
            "정밀도": round(precision_score(y_test, 예측_최적, zero_division=0), 3),
            "F1": round(f1_score(y_test, 예측_최적), 3),
        },
        {
            "기준": "F1",
            "설정값": F1_탐색.best_params_,
            "정확도(%)": round(F1_정확도, 2),
            "잡은 불량": F1_잡은불량,
            "헛경보": F1_헛경보,
            "재현율": round(recall_score(y_test, F1_예측_최적), 3),
            "정밀도": round(precision_score(y_test, F1_예측_최적, zero_division=0), 3),
            "F1": round(f1_score(y_test, F1_예측_최적), 3),
        },
    ]
).set_index("기준")

비교표

[1. 이번에 1등으로 뽑힌 설정값 (기준: F1)]
  {'max_depth': 10, 'min_samples_leaf': 1}

[2. 그 설정으로 시험용을 채점한 결과]
  정확도: 83.44 %
  잡은 불량: 8 / 헛경보: 39
  재현율: 0.381 정밀도: 0.17 F1: 0.235

[3. 재현율 기준 vs F1 기준]


,설정값,정확도(%),잡은 불량,헛경보,재현율,정밀도,F1
기준,,,,,,,
재현율,"{'max_depth': 3, 'min_samples_leaf': 20}",54.46,12,134,0.571,0.082,0.144
F1,"{'max_depth': 10, 'min_samples_leaf': 1}",83.44,8,39,0.381,0.170,0.235


---
## Step 5. 기준을 바꾸면 1등이 바뀐다

| 뽑은 기준 | 1등 설정 | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|---|
| 재현율 | [max_depth=3, min_samples_leaf=20] | [54.46]% | [12] | [134] | [0.571] | [0.144] |
| F1 | [max_depth=10, min_samples_leaf=10] | [83.44]% | [8] | [39] | [0.381] | [0.235] |

---
## Step 6. 내가 고른 설정

- 고른 기준 : [F1] - [놓친 불량도 줄이고 싶지만 헛경보 134건은 현장에서 감당이 안 될 것 같아서]
- 고른 설정 : [max_depth=10, min_samples_leaf=10]
- 이 설정의 시험용 성적 : [정확도 83.44% / 재현율 0.381 / 정밀도 0.170 / F1 0.235]

---
## 직접 해보기 (도전) - 다른 모델에도 다이얼이 있다

- 상황: 나무에만 다이얼이 있는 게 아니다
- 할 일: 로지스틱 회귀의 다이얼 하나를 네 값으로 돌려보고, 자동 탐색과 견줘본다
- 결과물: 네 줄짜리 표 1개 + 한 줄 메모

In [15]:
# 로지스틱 회귀도 사람이 정해줘야 하는 값(하이퍼파라미터)이 있다 - 그중 C를 네 값으로 돌려본다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

행들 = []
for C값 in [0.01, 0.1, 1, 10]:
    # C 자리만 바꾸고 나머지는 전부 같게 둔다
    로지스틱모델 = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C값, max_iter=1000, class_weight="balanced")
    )
    로지스틱모델.fit(X_train, y_train)
    로지스틱예측 = 로지스틱모델.predict(X_test)

    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 로지스틱예측).ravel()

    행들.append({
        "C": C값,
        "정확도(%)": round((로지스틱예측 == y_test).mean() * 100, 2),
        "잡은 불량": 잡은불량,
        "헛경보": 헛경보,
        "재현율": round(recall_score(y_test, 로지스틱예측), 3),
        "정밀도": round(precision_score(y_test, 로지스틱예측, zero_division=0), 3),
        "F1": round(f1_score(y_test, 로지스틱예측), 3),
    })

C값_비교표 = pd.DataFrame(행들).set_index("C")
C값_비교표

,정확도(%),잡은 불량,헛경보,재현율,정밀도,F1
C,,,,,,
0.01,74.84,12,70,0.571,0.146,0.233
0.10,75.48,10,66,0.476,0.132,0.206
1.00,75.16,9,66,0.429,0.120,0.188
10.00,75.48,10,66,0.476,0.132,0.206


### 저울 쪽 모델의 다이얼

| C | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|
| 0.01 | [74.84	]% | [12] | [70] | [0.571] | [0.233] |
| 0.1 | [75.48]% | [10] | [66] | [0.476] | [0.206] |
| 1 (기본값) | [75.16]% | [9] | [66] | [0.429] | [0.188] |
| 10 | [75.48]% | [10] | [66] | [0.476] | [0.206] |

- 알게 된 것 : [재현율로 보면 0.01이 제일 낫고, F1으로 봐도 0.01이 제일 낫다. <br>하지만 자에 따라 답이 갈릴 수 있다]